# MolForge Native AI Training
Downloads admin-only feedback, trains a baseline regressor, and exports weights.

In [ ]:
!pip -q install supabase pandas scikit-learn joblib


In [ ]:
import os, joblib, pandas as pd
from supabase import create_client
from sklearn.ensemble import RandomForestRegressor
url = os.environ['SUPABASE_URL']
key = os.environ['SUPABASE_SERVICE_KEY']
rows = create_client(url, key).table('predictions_feedback').select('*').execute().data
df = pd.DataFrame(rows).dropna(subset=['predicted_value'])
df['target'] = df['cloud_calculated_value'].fillna(df['mp_actual_value']).fillna(df['corrected_value'])
train = df.dropna(subset=['target'])
model = RandomForestRegressor(n_estimators=250, random_state=1337).fit(train[['predicted_value']], train['target'])
joblib.dump(model, 'molforge_feedback_calibrator.joblib')
print(f'Trained on {len(train)} feedback rows')
